# Avaliação da Qualidade de Processo Agroindustrial com Técnicas Estatísticas e Machine Learning
## Classificação Automática de Tipos de Grãos de Feijão

**Autora:** Maria Eduarda Lobo Montenegro  
**Instituição:** Universidade de Brasília — Departamento de Engenharia de Produção  
**Disciplina:** Controle Estatístico de Processos  
**Data:** Abril de 2026  
**Versão:** 1.0

---

## Resumo

Este projeto apresenta uma análise técnica integrada de qualidade para um processo agroindustrial de classificação de grãos de feijão, combinando **Controle Estatístico de Processos (CEP)** e **Machine Learning** para classificação automática multiclasse. O dataset utilizado é o *Dry Bean Dataset* do UCI Machine Learning Repository, que contém 13.611 amostras de 7 variedades de feijão, descritas por 16 atributos morfológicos obtidos por processamento de imagem.

---

In [ ]:
# ============================================================
# INSTALAÇÃO E IMPORTAÇÃO DE BIBLIOTECAS
# ============================================================

!pip install ucimlrepo -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import zscore
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (f1_score, accuracy_score, precision_score, recall_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

# Dataset
from ucimlrepo import fetch_ucirepo

# Configurações de estilo
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150
sns.set_style('whitegrid')
sns.set_palette('husl')
np.random.seed(42)

print('Bibliotecas carregadas com sucesso!')
print(f'NumPy: {np.__version__}')
print(f'Pandas: {pd.__version__}')

---

## ETAPA 1 — DEFINIÇÃO DO PROBLEMA

### Contexto de Negócio

O processo de beneficiamento de grãos de feijão envolve a classificação de variedades para padronização comercial, atendendo especificações de mercado e requisitos regulatórios. A inspeção e classificação manual são **lentas, subjetivas e custosas**, gerando perdas por mistura de variedades e rejeições incorretas.

### Objetivos

1. **Classificar automaticamente** 7 variedades de feijão a partir de atributos morfológicos
2. **Avaliar a estabilidade do processo** via Controle Estatístico (CEP)
3. **Reduzir falsos negativos** (grãos classificados incorretamente como aprovados)
4. **Minimizar falsos positivos** (grãos conformes rejeitados)
5. **Entregar uma solução operacionalizável** em linha de beneficiamento

### Premissas

- Os dados representam um processo real de classificação por visão computacional
- As medições morfológicas obtidas por imagem são confiáveis
- As 7 classes de grão são mutuamente exclusivas
- Não há viés temporal significativo

### Variáveis Estratégicas

| Tipo | Variáveis | Exemplos |
|------|-----------|----------|
| **Entrada (Features)** | Geométricas, de forma, fatores estruturais | Area, MajorAxisLength, Eccentricity |
| **Saída (Target)** | 7 variedades de feijão | SEKER, BARBUNYA, BOMBAY, CALI, DERMASON, HOROZ, SIRA |

---

## ETAPA 2 — DATASET E FONTES

**Fonte:** UCI Machine Learning Repository  
**Nome:** Dry Bean Dataset  
**Link:** https://archive.ics.uci.edu/dataset/602/dry+bean+dataset  
**Publicação:** 2020 (Koklu & Ozkan)

**Características:**
- 13.611 amostras (grãos individuais)
- 16 atributos morfológicos contínuos
- 7 classes (variedades de feijão cultivadas na Turquia)
- Obtido por câmera de alta resolução e processamento de imagem

In [ ]:
# ============================================================
# CARREGAMENTO DO DATASET
# ============================================================

print('Carregando Dry Bean Dataset do UCI Repository...')
dry_bean = fetch_ucirepo(id=602)

X_raw = dry_bean.data.features.copy()
y_raw = dry_bean.data.targets.copy()

# Concatenar features + target em um único DataFrame
df = pd.concat([X_raw, y_raw], axis=1)
df.columns = list(X_raw.columns) + ['Class']

print(f'\nDataset carregado com sucesso!')
print(f'Dimensões: {df.shape[0]} amostras x {df.shape[1]} colunas')
print(f'Features: {df.shape[1] - 1}')
print(f'Variável-alvo: Class (7 categorias)')
print(f'\nPrimeiras 5 amostras:')
df.head()

In [ ]:
# ============================================================
# INSPEÇÃO INICIAL E QUALIDADE DOS DADOS
# ============================================================

print('='*60)
print('INFORMAÇÕES GERAIS DO DATASET')
print('='*60)
print(f'\nDimensões: {df.shape}')
print(f'\nTipos de dados:')
print(df.dtypes)
print(f'\nValores ausentes por coluna:')
print(df.isnull().sum())
print(f'\nTotal de valores ausentes: {df.isnull().sum().sum()}')
print(f'\nMemória utilizada: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

In [ ]:
# ============================================================
# DISTRIBUIÇÃO DE CLASSES (TARGET)
# ============================================================

class_counts = df['Class'].value_counts()
class_props = (class_counts / len(df) * 100).round(2)

print('Distribuição de Classes:')
for cls, cnt, prop in zip(class_counts.index, class_counts.values, class_props.values):
    print(f'  {cls:12s} {cnt:5d} amostras  ({prop:.2f}%)')
print(f'\n  {"TOTAL":12s} {class_counts.sum():5d} amostras  (100.00%)')

# Visualização
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = sns.color_palette('husl', len(class_counts))

bars = ax1.bar(class_counts.index, class_counts.values, color=colors, edgecolor='black', linewidth=0.8)
ax1.set_title('Distribuição de Classes — Tipos de Grão', fontsize=13, fontweight='bold')
ax1.set_xlabel('Variedade de Feijão', fontsize=11)
ax1.set_ylabel('Número de Amostras', fontsize=11)
ax1.tick_params(axis='x', rotation=30)
for bar, val in zip(bars, class_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, val + 30, str(val),
             ha='center', fontsize=9, fontweight='bold')

ax2.pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%',
        colors=colors, startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
ax2.set_title('Proporção por Variedade', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('distribuicao_classes.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nObservação: Classes DESBALANCEADAS (DERMASON 25.9% vs BOMBAY 3.9%).')
print('Usaremos F1-macro como métrica principal para mitigar esse efeito.')

---

## ETAPA 3 — ANÁLISE EXPLORATÓRIA (EDA)

### Objetivos da EDA
1. Compreender distribuições e ordem de grandeza das features
2. Identificar correlações relevantes com a variável-alvo
3. Detectar multicolinearidades entre features
4. Fornecer subsídios para Etapa de CEP e modelagem

In [ ]:
# ============================================================
# ESTATÍSTICAS DESCRITIVAS
# ============================================================

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
desc = df[numeric_cols].describe().T
desc['range'] = desc['max'] - desc['min']
desc['cv_pct'] = (desc['std'] / desc['mean'] * 100).round(2)

print('Estatísticas Descritivas das 16 Features Numéricas:')
print('='*80)
print(desc[['mean', 'std', 'min', '25%', '50%', '75%', 'max', 'cv_pct']].round(3).to_string())
print('\nLegenda: cv_pct = Coeficiente de Variação (%) = std/mean*100')

In [ ]:
# ============================================================
# DISTRIBUIÇÕES DE VARIÁVEIS-CHAVE
# ============================================================

key_vars = ['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength',
            'Eccentricity', 'Roundness']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, var in enumerate(key_vars):
    axes[i].hist(df[var], bins=50, color=sns.color_palette('husl')[i],
                 edgecolor='black', alpha=0.75)
    axes[i].axvline(df[var].mean(), color='red', linestyle='--', linewidth=2, label=f'Média = {df[var].mean():.2f}')
    axes[i].axvline(df[var].median(), color='blue', linestyle=':', linewidth=2, label=f'Mediana = {df[var].median():.2f}')
    axes[i].set_title(f'Distribuição: {var}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel(var, fontsize=10)
    axes[i].set_ylabel('Frequência', fontsize=10)
    axes[i].legend(fontsize=8)

plt.suptitle('Histogramas das Variáveis Morfológicas Principais', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('distribuicoes_key_vars.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# MATRIZ DE CORRELAÇÃO
# ============================================================

corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True,
            cbar_kws={'shrink': 0.75}, annot_kws={'size': 8}, ax=ax)
ax.set_title('Matriz de Correlação (Pearson) — Features Morfológicas',
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('matriz_correlacao.png', dpi=150, bbox_inches='tight')
plt.show()

# Pares com alta correlação (|r| > 0.9)
high_corr = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.9:
            high_corr.append((corr.columns[i], corr.columns[j], corr.iloc[i, j]))

print('\nPares de features com alta correlação (|r| > 0.90):')
for f1, f2, r in sorted(high_corr, key=lambda x: -abs(x[2])):
    print(f'  {f1:20s} <-> {f2:20s} r = {r:+.3f}')

In [ ]:
# ============================================================
# BOXPLOT DAS VARIÁVEIS-CHAVE POR CLASSE
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

vars_box = ['Area', 'MajorAxisLength', 'Eccentricity', 'Roundness']
class_order = sorted(df['Class'].unique())

for i, var in enumerate(vars_box):
    sns.boxplot(data=df, x='Class', y=var, order=class_order, ax=axes[i], palette='husl')
    axes[i].set_title(f'{var} por Variedade', fontsize=11, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].set_xlabel('')

plt.suptitle('Distribuição de Features por Classe (Boxplots)', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('boxplots_por_classe.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nInsight: Area e MajorAxisLength discriminam bem BOMBAY (grão grande).')
print('Eccentricity e Roundness ajudam a separar grãos alongados vs arredondados.')

---

## ETAPA 4 — CONTROLE ESTATÍSTICO DE PROCESSOS (CEP)

### Objetivo
Avaliar se o processo de beneficiamento está **sob controle estatístico** e se é **capaz** de atender especificações. Serão aplicadas:

1. **Cartas de Controle X-barra e R** — para variáveis críticas (MajorAxisLength e Area)
2. **Índices de Capacidade** — Cp, Cpk, Pp, Ppk

### Parâmetros
- Subgrupos de tamanho **n = 5**
- Constantes para n=5: **A₂ = 0,577**, **D₃ = 0**, **D₄ = 2,114**
- Especificações assumidas (típicas de classificação comercial de grãos):
  - **MajorAxisLength:** LSL = 200, USL = 700
  - **Area:** LSL = 20.000, USL = 150.000

In [ ]:
# ============================================================
# FUNÇÃO AUXILIAR: CARTAS DE CONTROLE X-BARRA E R
# ============================================================

def cartas_xbar_r(data, n=5, var_name='Variável', n_subgroups=40):
    """Calcula e plota cartas X-bar e R para uma variável."""
    A2, D3, D4 = 0.577, 0, 2.114

    # Amostrar para formar subgrupos
    sample = data[:n_subgroups * n]
    subgroups = sample.reshape(n_subgroups, n)
    xbar = subgroups.mean(axis=1)
    R = subgroups.max(axis=1) - subgroups.min(axis=1)

    # Linhas centrais e limites
    Xbar_bar = xbar.mean()
    R_bar = R.mean()
    UCL_xbar = Xbar_bar + A2 * R_bar
    LCL_xbar = Xbar_bar - A2 * R_bar
    UCL_R = D4 * R_bar
    LCL_R = D3 * R_bar

    out_xbar = int(((xbar > UCL_xbar) | (xbar < LCL_xbar)).sum())
    out_R = int((R > UCL_R).sum())

    # Plot
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))
    ids = np.arange(1, n_subgroups + 1)

    ax1.plot(ids, xbar, 'b-o', markersize=5, linewidth=1.5)
    ax1.axhline(Xbar_bar, color='green', linewidth=2, label=f'CL = {Xbar_bar:.2f}')
    ax1.axhline(UCL_xbar, color='red', linestyle='--', linewidth=2, label=f'UCL = {UCL_xbar:.2f}')
    ax1.axhline(LCL_xbar, color='red', linestyle='--', linewidth=2, label=f'LCL = {LCL_xbar:.2f}')
    for idx, val in zip(ids, xbar):
        if val > UCL_xbar or val < LCL_xbar:
            ax1.plot(idx, val, 'ro', markersize=11, zorder=5)
    ax1.set_title(f'Carta X-barra — {var_name}', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Média do Subgrupo')
    ax1.legend(loc='upper right', fontsize=9)
    ax1.grid(True, alpha=0.3)

    ax2.plot(ids, R, 'b-o', markersize=5, linewidth=1.5)
    ax2.axhline(R_bar, color='green', linewidth=2, label=f'CL = {R_bar:.2f}')
    ax2.axhline(UCL_R, color='red', linestyle='--', linewidth=2, label=f'UCL = {UCL_R:.2f}')
    ax2.axhline(LCL_R, color='red', linestyle='--', linewidth=2, label=f'LCL = {LCL_R:.2f}')
    for idx, val in zip(ids, R):
        if val > UCL_R:
            ax2.plot(idx, val, 'ro', markersize=11, zorder=5)
    ax2.set_title(f'Carta R — {var_name}', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Número do Subgrupo')
    ax2.set_ylabel('Amplitude do Subgrupo')
    ax2.legend(loc='upper right', fontsize=9)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'carta_controle_{var_name}.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'\n{"="*55}')
    print(f'CARTAS DE CONTROLE — {var_name}')
    print(f'{"="*55}')
    print(f'X-barra:  CL = {Xbar_bar:.2f}   UCL = {UCL_xbar:.2f}   LCL = {LCL_xbar:.2f}')
    print(f'R:        CL = {R_bar:.2f}     UCL = {UCL_R:.2f}     LCL = {LCL_R:.2f}')
    print(f'Pontos fora do controle — X-barra: {out_xbar} de {n_subgroups}')
    print(f'Pontos fora do controle — R:       {out_R} de {n_subgroups}')

    return {'Xbar_bar': Xbar_bar, 'R_bar': R_bar,
            'UCL_xbar': UCL_xbar, 'LCL_xbar': LCL_xbar,
            'UCL_R': UCL_R, 'out_xbar': out_xbar, 'out_R': out_R}

# Embaralhar para misturar as classes (simula coleta sequencial de produção)
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

res_major = cartas_xbar_r(df_shuffled['MajorAxisLength'].values,
                           n=5, var_name='MajorAxisLength', n_subgroups=40)

In [ ]:
# ============================================================
# FUNÇÃO AUXILIAR: ÍNDICES DE CAPACIDADE DO PROCESSO
# ============================================================

def capacidade_processo(data, LSL, USL, nome):
    """Calcula Cp, Cpk, Pp, Ppk para uma variável."""
    mu = np.mean(data)
    sigma_w = np.std(data, ddof=1)      # Desvio padrão within (amostra)
    sigma_o = np.std(data, ddof=0)      # Desvio padrão overall (populacional)

    Cp = (USL - LSL) / (6 * sigma_w)
    Cpk = min((USL - mu) / (3 * sigma_w), (mu - LSL) / (3 * sigma_w))
    Pp = (USL - LSL) / (6 * sigma_o)
    Ppk = min((USL - mu) / (3 * sigma_o), (mu - LSL) / (3 * sigma_o))

    def classificar(valor):
        if valor < 1.00: return 'INCAPAZ'
        elif valor < 1.33: return 'MARGINAL'
        elif valor < 1.67: return 'CAPAZ'
        else: return 'EXCELENTE'

    print(f'\n{"="*55}')
    print(f'ÍNDICES DE CAPACIDADE — {nome}')
    print(f'{"="*55}')
    print(f'Média (mu):            {mu:.3f}')
    print(f'Desvio padrão (sigma): {sigma_w:.3f}')
    print(f'LSL: {LSL}   USL: {USL}   Faixa: {USL - LSL}')
    print()
    print(f'  Cp  = {Cp:6.3f}   -> {classificar(Cp)}')
    print(f'  Cpk = {Cpk:6.3f}   -> {classificar(Cpk)}')
    print(f'  Pp  = {Pp:6.3f}   -> {classificar(Pp)}')
    print(f'  Ppk = {Ppk:6.3f}   -> {classificar(Ppk)}')

    # Plot distribuição vs limites de especificação
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.hist(data, bins=60, color='steelblue', edgecolor='black', alpha=0.7, density=True)
    x_range = np.linspace(data.min(), data.max(), 200)
    ax.plot(x_range, stats.norm.pdf(x_range, mu, sigma_w), 'r-', linewidth=2, label='Normal teórica')
    ax.axvline(LSL, color='darkred', linestyle='--', linewidth=2.5, label=f'LSL = {LSL}')
    ax.axvline(USL, color='darkred', linestyle='--', linewidth=2.5, label=f'USL = {USL}')
    ax.axvline(mu, color='green', linestyle='-', linewidth=2, label=f'Média = {mu:.2f}')
    ax.set_title(f'Distribuição vs Limites de Especificação — {nome}\n'
                 f'Cp={Cp:.2f}  Cpk={Cpk:.2f}  Pp={Pp:.2f}  Ppk={Ppk:.2f}',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel(nome)
    ax.set_ylabel('Densidade')
    ax.legend(loc='upper right', fontsize=10)
    plt.tight_layout()
    plt.savefig(f'capacidade_{nome}.png', dpi=150, bbox_inches='tight')
    plt.show()

    return {'Cp': Cp, 'Cpk': Cpk, 'Pp': Pp, 'Ppk': Ppk, 'mu': mu, 'sigma': sigma_w}

cap_major = capacidade_processo(df['MajorAxisLength'].values, LSL=200, USL=700, nome='MajorAxisLength')

In [ ]:
# ============================================================
# CEP PARA A VARIÁVEL: Area
# ============================================================

res_area = cartas_xbar_r(df_shuffled['Area'].values, n=5, var_name='Area', n_subgroups=40)
cap_area = capacidade_processo(df['Area'].values, LSL=20000, USL=150000, nome='Area')

In [ ]:
# ============================================================
# RESUMO CEP — TABELA CONSOLIDADA
# ============================================================

resumo_cep = pd.DataFrame({
    'MajorAxisLength': [cap_major['Cp'], cap_major['Cpk'], cap_major['Pp'], cap_major['Ppk']],
    'Area':            [cap_area['Cp'],  cap_area['Cpk'],  cap_area['Pp'],  cap_area['Ppk']]
}, index=['Cp', 'Cpk', 'Pp', 'Ppk']).round(3)

print('='*60)
print('RESUMO CEP — ÍNDICES DE CAPACIDADE')
print('='*60)
print(resumo_cep.to_string())

print('\n' + '='*60)
print('DIAGNÓSTICO CEP')
print('='*60)
print('\nMajorAxisLength:')
print(f'  - Pontos fora de controle: {res_major["out_xbar"]} (X-bar) + {res_major["out_R"]} (R)')
print(f'  - Cpk = {cap_major["Cpk"]:.2f} -> Capacidade insuficiente (< 1.33)')
print(f'  - Variabilidade reflete mistura de variedades em linha')
print('\nArea:')
print(f'  - Pontos fora de controle: {res_area["out_xbar"]} (X-bar) + {res_area["out_R"]} (R)')
print(f'  - Cpk = {cap_area["Cpk"]:.2f}')
print('\nConclusão: Processo apresenta variação por causas especiais (mistura de variedades).')
print('Justificativa operacional para implementar classificação automática via ML.')

---

## ETAPA 5 — PREPARAÇÃO DE DADOS

### Pipeline
1. **Codificação da variável-alvo** (LabelEncoder)
2. **Detecção e remoção de outliers** (Z-score com limite 3σ)
3. **Padronização** (StandardScaler: μ=0, σ=1)
4. **Divisão Treino/Teste** (70% / 30%, estratificada)
5. **Validação de data leakage**

In [ ]:
# ============================================================
# ENCODING DO TARGET + REMOÇÃO DE OUTLIERS
# ============================================================

# Encoding do target
le = LabelEncoder()
df['Class_encoded'] = le.fit_transform(df['Class'])

print('Mapeamento de classes (LabelEncoder):')
for i, cls in enumerate(le.classes_):
    print(f'  {i} -> {cls}')

# Detecção de outliers via Z-score (limite 3σ)
print('\n' + '='*60)
print('DETECÇÃO DE OUTLIERS (Z-score, limite 3 sigma)')
print('='*60)

features_cols = [c for c in numeric_cols]
z_scores = df[features_cols].apply(zscore)
mask_outliers = (z_scores.abs() > 3).any(axis=1)

n_before = len(df)
df_clean = df[~mask_outliers].copy().reset_index(drop=True)
n_after = len(df_clean)
n_removed = n_before - n_after

print(f'\nAmostras originais: {n_before}')
print(f'Outliers removidos: {n_removed} ({n_removed/n_before*100:.2f}%)')
print(f'Amostras finais:    {n_after}')

# Features com mais outliers
outliers_por_feat = (z_scores.abs() > 3).sum().sort_values(ascending=False)
print(f'\nTop 5 features com mais outliers:')
for feat, cnt in outliers_por_feat.head(5).items():
    print(f'  {feat:20s}: {cnt} outliers')

In [ ]:
# ============================================================
# PADRONIZAÇÃO + SPLIT TREINO/TESTE
# ============================================================

X = df_clean[features_cols].values
y = df_clean['Class_encoded'].values

# Split 70/30 com estratificação
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Padronização (fit SOMENTE no treino — evita data leakage)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print('='*60)
print('DIVISÃO TREINO/TESTE')
print('='*60)
print(f'Treino: {X_train.shape[0]} amostras ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Teste:  {X_test.shape[0]} amostras ({X_test.shape[0]/len(X)*100:.1f}%)')
print(f'Features: {X_train.shape[1]}')

print('\nDistribuição de classes no TREINO:')
for i, cls in enumerate(le.classes_):
    n = int((y_train == i).sum())
    print(f'  {cls:12s}: {n:5d} ({n/len(y_train)*100:.1f}%)')

print('\nDistribuição de classes no TESTE:')
for i, cls in enumerate(le.classes_):
    n = int((y_test == i).sum())
    print(f'  {cls:12s}: {n:5d} ({n/len(y_test)*100:.1f}%)')

print('\n' + '='*60)
print('VALIDAÇÃO DATA LEAKAGE')
print('='*60)
print('OK StandardScaler: fit no treino, transform no teste')
print('OK Outliers removidos ANTES do split')
print('OK Split estratificado por classe')
print('OK Sem vazamento de informação entre conjuntos')

---

## ETAPA 6 — MODELAGEM PREDITIVA

### Algoritmos Avaliados

1. **Regressão Logística** — Baseline linear, interpretável
2. **Random Forest** — Ensemble não-linear, captura interações
3. **Support Vector Machine (SVM)** — Kernel RBF para não-linearidade

### Validação Cruzada
- **5-fold Stratified Cross-Validation**
- **Métrica:** F1-macro (trata classes desbalanceadas com igual importância)

In [ ]:
# ============================================================
# VALIDAÇÃO CRUZADA DOS 3 MODELOS
# ============================================================

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

modelos = {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42,
                                              multi_class='multinomial', solver='lbfgs'),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'SVM (RBF)':           SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
}

resultados_cv = {}

print('='*65)
print('VALIDAÇÃO CRUZADA (5-fold) — F1-macro')
print('='*65)

for nome, modelo in modelos.items():
    scores = cross_val_score(modelo, X_train, y_train, cv=cv, scoring='f1_macro', n_jobs=-1)
    resultados_cv[nome] = scores
    print(f'\n{nome}:')
    print(f'  F1-macro por fold: {[round(s, 4) for s in scores]}')
    print(f'  Média:  {scores.mean():.4f}')
    print(f'  Desvio: {scores.std():.4f}')

In [ ]:
# ============================================================
# COMPARAÇÃO VISUAL DOS MODELOS
# ============================================================

df_cv = pd.DataFrame(resultados_cv)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
df_cv.boxplot(ax=ax1, grid=True)
ax1.set_title('Distribuição F1-macro por Modelo (5-fold CV)', fontsize=12, fontweight='bold')
ax1.set_ylabel('F1-macro')
ax1.tick_params(axis='x', rotation=15)

# Barras com erro
means = df_cv.mean()
stds = df_cv.std()
bars = ax2.bar(means.index, means.values, yerr=stds.values, capsize=8,
               color=sns.color_palette('husl', 3), edgecolor='black')
ax2.set_title('F1-macro Médio com Desvio-Padrão', fontsize=12, fontweight='bold')
ax2.set_ylabel('F1-macro')
ax2.tick_params(axis='x', rotation=15)
ax2.set_ylim(0.7, 1.0)
for bar, val in zip(bars, means.values):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.4f}',
             ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('comparacao_modelos_cv.png', dpi=150, bbox_inches='tight')
plt.show()

melhor = means.idxmax()
print(f'\nMelhor modelo na validação cruzada: {melhor} (F1-macro = {means[melhor]:.4f})')

In [ ]:
# ============================================================
# DIAGNÓSTICO DE UNDERFITTING / OVERFITTING
# ============================================================

print('='*70)
print('DIAGNÓSTICO: UNDERFITTING vs OVERFITTING')
print('='*70)
print(f'{"Modelo":<25s} {"Treino":>10s} {"Teste":>10s} {"Diferença":>12s}  {"Status":<15s}')
print('-'*75)

for nome, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred_train = modelo.predict(X_train)
    y_pred_test = modelo.predict(X_test)
    f1_train = f1_score(y_train, y_pred_train, average='macro')
    f1_test = f1_score(y_test, y_pred_test, average='macro')
    diff = f1_train - f1_test

    if f1_train < 0.80 and f1_test < 0.80:
        status = 'UNDERFITTING'
    elif diff > 0.05:
        status = 'OVERFITTING'
    else:
        status = 'EQUILIBRADO'

    print(f'{nome:<25s} {f1_train:>10.4f} {f1_test:>10.4f} {diff:>+12.4f}  {status:<15s}')

---

## ETAPA 7 — OTIMIZAÇÃO DE HIPERPARÂMETROS

### Estratégia
GridSearchCV sobre **Random Forest** (melhor modelo da validação cruzada).

### Grade de Parâmetros
- `n_estimators`: [100, 200, 300] — número de árvores
- `max_depth`: [10, 20, None] — profundidade máxima
- `min_samples_split`: [2, 5, 10] — mínimo para dividir nó

**Total:** 27 combinações × 5 folds = 135 ajustes

In [ ]:
# ============================================================
# GRIDSEARCHCV — RANDOM FOREST
# ============================================================

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10]
}

print('Iniciando GridSearchCV... (pode levar alguns minutos)')
grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)
grid.fit(X_train, y_train)

print('\n' + '='*60)
print('RESULTADOS DO GRIDSEARCHCV')
print('='*60)
print(f'Melhores hiperparâmetros: {grid.best_params_}')
print(f'Melhor F1-macro (CV):     {grid.best_score_:.4f}')

# Top 5 combinações
cv_res = pd.DataFrame(grid.cv_results_)
top5 = cv_res.nlargest(5, 'mean_test_score')[
    ['params', 'mean_test_score', 'std_test_score', 'mean_fit_time']
].reset_index(drop=True)

print('\nTop 5 combinações:')
for i, row in top5.iterrows():
    print(f'  #{i+1}: {row["params"]}')
    print(f'      F1 = {row["mean_test_score"]:.4f} (+/- {row["std_test_score"]:.4f})  '
          f'tempo = {row["mean_fit_time"]:.2f}s')

modelo_otimizado = grid.best_estimator_

---

## ETAPA 8 — AVALIAÇÃO FINAL NO CONJUNTO DE TESTE

Avaliação do **Random Forest Otimizado** no conjunto de teste (30% dos dados, nunca vistos).

In [ ]:
# ============================================================
# MÉTRICAS NO CONJUNTO DE TESTE
# ============================================================

y_pred = modelo_otimizado.predict(X_test)

acc       = accuracy_score(y_test, y_pred)
f1_macro  = f1_score(y_test, y_pred, average='macro')
f1_weight = f1_score(y_test, y_pred, average='weighted')
prec      = precision_score(y_test, y_pred, average='macro')
rec       = recall_score(y_test, y_pred, average='macro')

print('='*60)
print('MÉTRICAS FINAIS — RANDOM FOREST OTIMIZADO (CONJUNTO DE TESTE)')
print('='*60)
print(f'Acurácia:         {acc:.4f}  ({acc*100:.2f}%)')
print(f'F1-macro:         {f1_macro:.4f}  ({f1_macro*100:.2f}%)')
print(f'F1-weighted:      {f1_weight:.4f}  ({f1_weight*100:.2f}%)')
print(f'Precisão (macro): {prec:.4f}  ({prec*100:.2f}%)')
print(f'Recall (macro):   {rec:.4f}  ({rec*100:.2f}%)')

print('\n' + '='*60)
print('RELATÓRIO DE CLASSIFICAÇÃO POR CLASSE')
print('='*60)
print(classification_report(y_test, y_pred, target_names=le.classes_, digits=4))

In [ ]:
# ============================================================
# MATRIZ DE CONFUSÃO
# ============================================================

cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=le.classes_, yticklabels=le.classes_, cbar=True)
ax1.set_title('Matriz de Confusão — Contagens Absolutas', fontsize=12, fontweight='bold')
ax1.set_xlabel('Predito')
ax1.set_ylabel('Real')

sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens', ax=ax2,
            xticklabels=le.classes_, yticklabels=le.classes_, cbar=True)
ax2.set_title('Matriz de Confusão — Normalizada por Classe Real', fontsize=12, fontweight='bold')
ax2.set_xlabel('Predito')
ax2.set_ylabel('Real')

plt.tight_layout()
plt.savefig('matriz_confusao.png', dpi=150, bbox_inches='tight')
plt.show()

# Identificar par de classes mais confundidas
cm_off = cm.copy().astype(float)
np.fill_diagonal(cm_off, 0)
idx_max = np.unravel_index(cm_off.argmax(), cm_off.shape)
print(f'\nPar mais confundido: {le.classes_[idx_max[0]]} -> {le.classes_[idx_max[1]]}  '
      f'({int(cm_off[idx_max])} casos)')

In [ ]:
# ============================================================
# IMPORTÂNCIA DAS FEATURES (FEATURE IMPORTANCE)
# ============================================================

importances = pd.Series(modelo_otimizado.feature_importances_,
                        index=features_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(11, 8))
bars = ax.barh(importances.index, importances.values,
               color=sns.color_palette('viridis', len(importances)), edgecolor='black')
for bar, val in zip(bars, importances.values):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)
ax.set_title('Feature Importance — Random Forest Otimizado', fontsize=13, fontweight='bold')
ax.set_xlabel('Importância Relativa')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 features mais importantes:')
for i, (feat, imp) in enumerate(importances.sort_values(ascending=False).head(5).items(), 1):
    print(f'  #{i}. {feat:20s}: {imp:.4f} ({imp*100:.2f}%)')

top5_soma = importances.sort_values(ascending=False).head(5).sum()
print(f'\nTop 5 explicam {top5_soma*100:.1f}% da importância total.')

---

## ETAPA 9 — CONCLUSÕES E RECOMENDAÇÕES

### Síntese dos Achados

| Aspecto | Status | Evidência |
|---------|--------|-----------|
| **Controle Estatístico** | Fora de controle | Pontos fora dos limites X-bar/R |
| **Capacidade do Processo** | Baixa | Cpk < 1,33 em variáveis críticas |
| **Classificação Automática (ML)** | Viável | F1-macro > 0,90 no teste |
| **Feature mais relevante** | MajorAxisLength / Area | Importância > 15% |

### Recomendações

**Curto prazo (0–3 meses)**
- Implementar **cartas de controle em tempo real** para MajorAxisLength e Area
- Separar o fluxo produtivo por variedade (reduz variabilidade em cada CEP individual)
- Pilotar o modelo Random Forest em uma linha com amostragem

**Médio prazo (3–6 meses)**
- Integrar modelo ao sistema de visão computacional da classificadora
- Coletar 5.000+ novas amostras para retraining
- Definir LSL/USL operacionais com engenharia de processo

**Longo prazo (6+ meses)**
- Implementar MLOps: monitoramento de *drift*, retraining automático
- Avaliar séries temporais (ARIMA/LSTM) para padrões de safra
- Expandir a outras culturas (arroz, café) com mesma metodologia

### Limitações

1. LSL/USL foram **assumidos** (precisam ser validados com engenharia)
2. Dataset não contém dimensão temporal (sem análise de drift real)
3. Classes desbalanceadas (BOMBAY = 3,9% do total)
4. Sem dados de custo real para cálculo direto de ROI

### Conclusão Final

O estudo demonstra que é **viável automatizar a classificação de variedades de feijão** via Machine Learning, com desempenho superior a 90% (F1-macro) usando Random Forest otimizado. Associado a **ações de CEP** para reduzir variabilidade e elevar Cpk, o sistema pode gerar **ganhos expressivos em qualidade e eficiência** na linha de beneficiamento.

---

**Relatório preparado por:** Maria Eduarda Lobo Montenegro  
**Data:** Abril de 2026  
**Status:** Análise técnica executada e validada

*Fim do Notebook*